In [ ]:
def reset_filepath(data_scenario):
    reset_file_tas = [f for f in os.listdir(data_scenario) if f.startswith('reset_sliced_detrended_tas')][0]
    reset_file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('reset_sliced_detrended_prsn')][0]
    reset_file_pr = [f for f in os.listdir(data_scenario) if f.startswith('reset_sliced_detrended_pr_')][0]
    return data_scenario + '/' + reset_file_tas, data_scenario + '/' + reset_file_prsn, data_scenario + '/' + reset_file_pr

In [ ]:
# def select_area(data_dictionary, min_lon, max_lon, min_lat, max_lat, plot = 0):
#     dictionary = {}
#     for scenario in data_dictionary:
#         dictionary[scenario] = {}
#         for parameter in data_dictionary[scenario]:
#             dictionary[scenario][parameter] = data_dictionary[scenario][parameter].sel(lat = slice(min_lat, max_lat)).sel(lon = slice(min_lon, max_lon))

#     latitudes = np.concatenate([np.linspace(min_lat, max_lat), np.zeros(50) + min_lat, np.linspace(min_lat, max_lat), np.zeros(50) + max_lat])
#     longitudes = np.concatenate([np.zeros(50) + min_lon, np.linspace(min_lon, max_lon), np.zeros(50) + max_lon,  np.linspace(min_lon, max_lon)])
    
#     if plot == 1:
#         plot_locations(longitudes, latitudes, 0.8)

#     return dictionary

In [ ]:
def filepath(data_scenario, months, detrend = 'detrended'):
    if months == 1:
        if detrend == 'detrended': # (for Data_analysis)
            file_tas = [f for f in os.listdir(data_scenario) if f.startswith('detrended_tas')][0]
            file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('detrended_prsn')][0]
            file_pr = [f for f in os.listdir(data_scenario) if f.startswith('detrended_pr.')][0]
            file_snfr = [f for f in os.listdir(data_scenario) if f.startswith('detrended_snfr')][0]
            
        elif detrend == 'sliced':    # data not detrended but duplicate years removed and Prsn, Pr < e-3 = 0 (for Snfr_Temp fitting)
            file_tas = [f for f in os.listdir(data_scenario) if f.startswith('sliced_tas')][0]
            file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('sliced_prsn')][0]
            file_pr = [f for f in os.listdir(data_scenario) if f.startswith('sliced_pr.')][0]
            file_snfr = [f for f in os.listdir(data_scenario) if f.startswith('sliced_snfr')][0]
            
        elif detrend == 'raw':      # raw data   (For Data_preprocessing)
            file_tas = [f for f in os.listdir(data_scenario) if f.startswith('tas_')][0]
            file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('prsn_')][0]
            file_pr = [f for f in os.listdir(data_scenario) if f.startswith('pr_')][0]
            return data_scenario + '/' + file_tas, data_scenario + '/' + file_prsn, data_scenario + '/' + file_pr, 
        else:
            print('Specify dataset (detrended, sliced or raw)')

    elif months == 12:
        if detrend == 'detrended':
            file_tas = [f for f in os.listdir(data_scenario) if f.startswith('detrended_yearly_tas')][0]
            file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('detrended_yearly_prsn')][0]
            file_pr = [f for f in os.listdir(data_scenario) if f.startswith('detrended_yearly_pr.')][0]
            file_snfr = [f for f in os.listdir(data_scenario) if f.startswith('detrended_yearly_snfr')][0]
        else:      # yearly means, calculated from sliced data
            file_tas = [f for f in os.listdir(data_scenario) if f.startswith('yearly_tas')][0]
            file_prsn = [f for f in os.listdir(data_scenario) if f.startswith('yearly_prsn')][0]
            file_pr = [f for f in os.listdir(data_scenario) if f.startswith('yearly_pr.')][0]
            file_snfr = [f for f in os.listdir(data_scenario) if f.startswith('yearly_snfr')][0]
    else:
        print('specify yearly or monthly data')
    return data_scenario + '/' + file_tas, data_scenario + '/' + file_prsn, data_scenario + '/' + file_pr, data_scenario + '/' + file_snfr

### Fixing the startdates

In [ ]:
def set_startdate(data_scenario):
    reset_files = []
    for file in sliced_files(data_scenario):
        reset_file = 'reset_'+file
        reset_files.append(data_scenario + '/' + reset_file)
        cdo.settaxis('1850-01-16,12:00:00,30day',
            input= data_scenario + '/' + file, output= data_scenario + '/' + reset_file)
    return reset_files

file_reset_tas_pi, file_reset_prsn_pi, file_reset_pr_pi = set_startdate(data_pi)
file_reset_tas_2K, file_reset_prsn_2K, file_reset_pr_2K = set_startdate(data_2K)
file_reset_tas_4K, file_reset_prsn_4K, file_reset_pr_4K = set_startdate(data_4K)

In [ ]:
def yearmean(data_scenario):
    yearmean_files = []
    for file in sliced_files(data_scenario):
        yearmean_file = 'reset_yearmean_'+file
        yearmean_files.append(data_scenario + '/' + yearmean_file)
        cdo.settaxis('1850-06-16,12:00:00,365day',
             input= cdo.yearmean(input = data_scenario + '/' + file), output= data_scenario + '/' + yearmean_file)
    return yearmean_files

In [ ]:
file_reset_tas_pi, file_reset_prsn_pi, file_reset_pr_pi = reset_filepath(data_pi)
file_reset_tas_2K, file_reset_prsn_2K, file_reset_pr_2K = reset_filepath(data_2K)
file_reset_tas_4K, file_reset_prsn_4K, file_reset_pr_4K = reset_filepath(data_4K)

In [ ]:
# If I want to get the detrended data, withot the averaged being removed I can use cdo trend to obtain the linear fit parameters.
# I can then add the intersept to obtain the original data

# does not work yet
cdo.trend(input = cdo.seltimestep('1/3600', input = data_pi + '/' + files(data_pi)[0]),
          output = data_pi + '/' + 'trend1_' + files(data_pi)[0], output2 = data_pi + '/' + 'trend2_' + files(data_pi)[0])

trend_tas_pi = xr.open_dataset(data_pi + '/' + 'trend1.nc')
trend_tas_pi.tas.values

### Soooo isel selects part of an xarray by index, sel selects it based on index. 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
import numpy as np
import matplotlib.animation as animation

# Select the variable
tas = tas_pi['tas']

# Choose the projection (PlateCarree = normal lat/lon map)
proj = ccrs.PlateCarree()

# Create figure
fig = plt.figure(figsize=(10, 4))
ax = plt.axes(projection=proj)
ax.coastlines()

# First frame (time=0)
frame0 = tas.isel(time=0)
img = frame0.plot(
    ax=ax,
    transform=proj,
    add_colorbar=False,
    cmap='coolwarm'
)

# Update function for each time step
def update(i):
    ax.clear()
    ax.coastlines()
    frame = tas.isel(time=i)
    img = frame.plot(
        ax=ax,
        transform=proj,
        add_colorbar=False,
        cmap='coolwarm'
    )
    ax.set_title(f"Temperature at time index {i}")
    return img

# Create animation
anim = FuncAnimation(fig, update, frames=100, interval=150)

# Display animation in Jupyter
from IPython.display import HTML
HTML(anim.to_jshtml())
anim.save("demo_animation.mp4", writer=animation.FFMpegWriter(fps=10))

In [ ]:
## This part of the code should only be run once, to create the correct files
# This is for if I want to keep the values of snfr if there is no precipitation


# Calculate the fraction between snow and precipitation and make values higher than 1.1 0, to take out instances of zero precipitation
def snow_fraction(data_scenario):
    file_tas_sliced, file_prsn_sliced, file_pr_sliced = sliced_filepath(data_scenario)    
    cdo.setrtoc("-1e99,0,0", input=cdo.setrtoc('1.01,1e99,0', input=cdo.div(input = f'{file_prsn_sliced} {file_pr_sliced}')), output = f'{data_scenario}/snow_fraction.nc')
    return f'{data_scenario}/snow_fraction.nc'

file_snow_fraction_pi = snow_fraction(data_pi)
file_snow_fraction_2K = snow_fraction(data_2K)
file_snow_fraction_4K = snow_fraction(data_4K)

In [ ]:
# neppe data om als test te gebruiken voor functie detrend

import numpy as np
import xarray as xr

# --- Define dimensions ---
time = np.arange(1980, 2021)            # 41 years
lat = np.linspace(-90, 90, 10)          # 10 lat points
lon = np.linspace(0, 360, 20)           # 20 lon points

coords = {"time": time, "lat": lat, "lon": lon}

# --- Create synthetic linear trend ---
slope = 0.2          # units per year
intercept = 10.0

# Proper broadcasting: (time, lat, lon)
linear_trend = (
    slope * (time - time[0])[:, None, None]     # (time, 1, 1)
    + intercept                                  # scalar
    + np.zeros((1, len(lat), len(lon)))          # (1, lat, lon)
)

# Add noise of correct shape
rng = np.random.default_rng(42)
noise = rng.normal(scale=0.5, size=linear_trend.shape)

data = linear_trend + noise

# --- Create DataArray ---
da = xr.DataArray(
    data,
    coords=coords,
    dims=("time", "lat", "lon"),
    name="synthetic_variable",
)


In [ ]:
def fitting_snfr_T(tas_array, snfr_array):
    a = []
    b = []
    for latitude in tas_array.lat.values:
        print(latitude)
        
        for longitude in tas_array.lon.values:
                
            tas_1d = tas_array.sel(lat = latitude).sel(lon = longitude)
            snfr_1d = snfr_array.sel(lat = latitude).sel(lon = longitude)
            
            mask = (~tas_1d.isnull()) & (~snfr_1d.isnull())
            
            tas_1d  = tas_1d.where(mask, drop=True).values
            snfr_1d = snfr_1d.where(mask, drop=True).values  

            if (snfr_1d == 0).all():
                continue

            elif (snfr_1d ==1).all():
                continue

            else:
                try:
                    popt, pcov = curve_fit(snowfraction, tas_1d, snfr_1d)
                except RuntimeError:
                    # curve_fit failed — skip this iteration
                    # print("Fit failed for this iteration, skipping.")
                    continue
        
                a.append(popt[0])
                b.append(popt[1])

    return np.mean(a), np.mean(b)

In [ ]:
def fit_wrapper(x, y):
    from scipy.optimize import curve_fit
    try:
        popt, _ = curve_fit(myfunc, x, y)
        return popt
    except Exception:
        # Return NaNs for failed fits
        return np.array([np.nan, np.nan, np.nan])

params = xr.apply_ufunc(
    fit_wrapper,
    tas_pi,
    snfr_pi,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["param"]],
    vectorize=True,
    output_dtypes=[float],
    dask="parallelized",
    output_sizes={"param": 2},   # number of fitted parameters
)


### Relative (normalized) Variablility

#### Comparison relative (normalized) yearly variability

In [ ]:
compared_relative_variability_tas_2K = relative_variability_tas_2K - relative_variability_tas_pi
compared_relative_variability_tas_4K = relative_variability_tas_4K - relative_variability_tas_pi

compared_relative_variability_pr_2K = relative_variability_pr_2K - relative_variability_pr_pi
compared_relative_variability_pr_4K = relative_variability_pr_4K - relative_variability_pr_pi

compared_relative_variability_prsn_2K = relative_variability_prsn_2K - relative_variability_prsn_pi
compared_relative_variability_prsn_4K = relative_variability_prsn_4K - relative_variability_prsn_pi

compared_relative_variability_snfr_2K = relative_variability_snfr_2K - relative_variability_snfr_pi
compared_relative_variability_snfr_4K = relative_variability_snfr_4K - relative_variability_snfr_pi

In [ ]:

def plot_variability_polar(variabilities, titles, title2, savename, max_scale = 'no_max', min_scale = 0, bar_label = 'Temperature variability (K)', colors="YlOrRd", max_scale2 = 'no_max', min_scale2 = 0, colors2 = 'coolwarm'):
    proj = ccrs.NorthPolarStereo()

    # --- Figure + GridSpec layout ---
    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(2, 9)

    ax_top = fig.add_subplot(gs[0, 1:6], projection=proj)
    ax_bl = fig.add_subplot(gs[1, 0:4], projection=proj)
    ax_br = fig.add_subplot(gs[1, 4:8], projection=proj)

    axes = [ax_top, ax_bl, ax_br]

    # --- Circular clipping helper ---
    def set_circular_boundary(ax):
        theta = np.linspace(0, 2*np.pi, 100)
        center, radius = [0.5, 0.5], 0.5
        verts = np.vstack([np.sin(theta), np.cos(theta)]).T * radius + center
        circle = mpath.Path(verts)
        ax.set_boundary(circle, transform=ax.transAxes)

    # -----------------------------
    # TOP PANEL – own colorbar
    # -----------------------------
    if max_scale == "no_max":
        mesh0 = variabilities[0].plot(
            ax=ax_top,
            transform=ccrs.PlateCarree(),
            cmap=colors,
            add_colorbar=False
        )
    else:
        mesh0 = variabilities[0].plot(
            ax=ax_top,
            transform=ccrs.PlateCarree(),
            vmin=min_scale,
            vmax=max_scale,
            cmap=colors,
            add_colorbar=False
        )

    ax_top.set_title(titles[0])

    ax_top.coastlines()
    ax_top.add_feature(cfeature.BORDERS, linestyle=":")
    ax_top.set_extent([-180, 180, 60, 90], crs=ccrs.PlateCarree())
    set_circular_boundary(ax_top)

    cbar0 = fig.colorbar(mesh0, ax=ax_top, fraction=0.035, pad=0.03)
    cbar0.set_label(bar_label)

    # ---------------------------------
    # BOTTOM TWO — shared diverging CBAR
    # ---------------------------------

    if max_scale2 == 'no_max':
        maxabs = np.nanmax(np.abs(np.concatenate([
            variabilities[1].values.flatten(),
            variabilities[2].values.flatten()
        ])))

        divnorm = mcolors.TwoSlopeNorm(vmin=-maxabs, vcenter=0, vmax=maxabs)
        max_scale2 = maxabs
        min_scale2 = -maxabs

    else:
        divnorm = mcolors.TwoSlopeNorm(vmin=min_scale2, vcenter=0, vmax=max_scale2)

    shared_mesh = None

    for ax, var, title in zip(axes[1:], variabilities[1:], titles[1:]):

        shared_mesh = var.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=colors2,
            norm=divnorm,
            add_colorbar=False
        )

        ax.set_title(title)
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=":")
        ax.set_extent([-180, 180, 60, 90], crs=ccrs.PlateCarree())
        set_circular_boundary(ax)

    # --- Shared colorbar ---
    cbar_shared = fig.colorbar(
        shared_mesh,
        ax=[ax_bl, ax_br],
        fraction=0.035,
        pad=0.3,
        panchor = (-0.05, 0.5),
        orientation = 'horizontal',
        ticks = list(np.round(np.linspace(min_scale2, min_scale2/2, 2),2)) + list(np.round(np.linspace(0, max_scale2, 3),2))
    )
    cbar_shared.set_label(title2)

    # --- Final layout ---
    plt.subplots_adjust(hspace=-0.01, wspace=0.8)

    plt.savefig(savename, bbox_inches="tight", dpi=300)
    plt.show()


In [ ]:
titles_norm_changes = [[r'Normalized pre-industrial $\sigma_{T}$ (K)', r'$\Delta \sigma_{T}$ 2K warming (K)', r'$\Delta \sigma_{T}$ 4K warming (K)'],
          [r'Normalized pre-industrial $\sigma_{pr}$ (mm/day)', r'$\Delta \sigma_{pr}$ 2K warming (mm/day)', r'$\Delta \sigma_{pr}$ 4K warming (mm/day)'],
          [r'Normalized pre-industrial $\sigma_{prsn}$ (mm/day)', r'$\Delta \sigma_{prsn}$ 2K warming (mm/day)', r'$\Delta \sigma_{prsn}$ 4K warming (mm/day)'],
          [r'Normalized pre-industrial $\sigma_{f}$', r'$\Delta \sigma_{f}$ 2K warming ', r'$\Delta \sigma_{f}$ 4K warming']]


plot_variability_polar([relative_variability_tas_pi, compared_relative_variability_tas_2K, compared_relative_variability_tas_4K], titles_norm_changes[0], r'$\Delta \sigma_{T}$ (K)', 'comparison_relative_variability_tas.png', 'no_max', 0,'Temperature variability (K)')
plot_variability_polar([relative_variability_pr_pi, compared_relative_variability_pr_2K, compared_relative_variability_pr_4K], titles_norm_changes[1], r'$\Delta \sigma_{pr}$ (mm/day)', 'comparison_relative_variability_pr.png', 0.36, 0.05,'Presipitation variability (mm/day)', 'PuBu', 0.17, -0.17)
plot_variability_polar([relative_variability_prsn_pi, compared_relative_variability_prsn_2K, compared_relative_variability_prsn_4K], titles_norm_changes[2], r'$\Delta \sigma_{prsn}$ (mm/day)', 'comparison_relative_variability_prsn.png', 0.5, 0.13,'Snowfall variability (mm/day)', 'YlGnBu', 1.2,-0.05)
plot_variability_polar([relative_variability_snfr_pi, compared_relative_variability_snfr_2K, compared_relative_variability_snfr_4K], titles_norm_changes[3], r'$\Delta \sigma_{f}$', 'comparison_relative_variability_snfr.png', 0.45, 0,'Snowfall fraction variability', 'BuGn', 1.2, -0.05)

#### Calculating contributions to snowfall variability

In [ ]:
# Same data handeling as in function for obtaining snow fraction to not 
def new_pr_and_prsn(pr, prsn):
    pr_new = pr.where(pr >= 0)
    prsn_new = prsn.where(pr >= 0)
    
    prsn_new = prsn.where(prsn_new >= 0, 0)
    
    difference = pr_new-prsn_new
    prsn_overshot = difference.where(difference < 0, 0)
    print(prsn_overshot.min().values)
    prsn_new = prsn_new + prsn_overshot # make it so snowfall cannot be larger than precipitation
    return pr_new, prsn_new

# creating corrected data arrays
corrected_pr_pi, corrected_prsn_pi = new_pr_and_prsn(pr_pi_yearly, prsn_pi_yearly)
corrected_pr_2K, corrected_prsn_2K = new_pr_and_prsn(pr_2K_yearly, prsn_2K_yearly)
corrected_pr_4K, corrected_prsn_4K = new_pr_and_prsn(pr_4K_yearly, prsn_4K_yearly)

# calculating the means
mean_snfr_pi = snfr_pi_yearly.mean(dim = 'time')
mean_pr_pi = pr_pi_yearly.mean(dim = 'time')
mean_snfr_2K = snfr_2K_yearly.mean(dim = 'time')
mean_pr_2k = pr_2K_yearly.mean(dim = 'time')
mean_snfr_4K = snfr_4K_yearly.mean(dim = 'time')
mean_pr_4K = pr_4K_yearly.mean(dim = 'time')

corrected_mean_pr_pi = corrected_pr_pi.mean(dim = 'time')
corrected_mean_pr_2K = corrected_pr_pi.mean(dim = 'time')
corrected_mean_pr_4K = corrected_pr_pi.mean(dim = 'time')

corrected_mean_prsn_pi = corrected_prsn_pi.mean(dim = 'time')
corrected_mean_prsn_2K = corrected_prsn_pi.mean(dim = 'time')
corrected_mean_prsn_4K = corrected_prsn_pi.mean(dim = 'time')

# calculating the variability
corrected_variability_pr_pi = corrected_pr_pi.std(dim = 'time').sel(lat = slice(20,90))
corrected_variability_prsn_pi = corrected_prsn_pi.std(dim = 'time').sel(lat = slice(20,90))

corrected_variability_pr_2K = corrected_pr_2K.std(dim = 'time').sel(lat = slice(20,90))
corrected_variability_prsn_2K = corrected_prsn_2K.std(dim = 'time').sel(lat = slice(20,90))

corrected_variability_pr_4K = corrected_pr_4K.std(dim = 'time').sel(lat = slice(20,90))
corrected_variability_prsn_4K = corrected_prsn_4K.std(dim = 'time').sel(lat = slice(20,90))

# calculating the relative (normalized variability)
corrected_relative_variability_pr_pi = corrected_pr_pi.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_pr_pi
corrected_relative_variability_prsn_pi = corrected_prsn_pi.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_prsn_pi

corrected_relative_variability_pr_2K = corrected_pr_2K.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_pr_2K
corrected_relative_variability_prsn_2K = corrected_prsn_2K.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_prsn_2K

corrected_relative_variability_pr_4K = corrected_pr_4K.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_pr_4K
corrected_relative_variability_prsn_4K = corrected_prsn_4K.std(dim = 'time').sel(lat = slice(20,90))/corrected_mean_prsn_4K


#covariance
covariance_pi = xr.cov(snfr_pi_yearly, pr_pi_yearly, dim = 'time')
covariance_2K = xr.cov(snfr_2K_yearly, pr_2K_yearly, dim = 'time')
covariance_4K = xr.cov(snfr_4K_yearly, pr_4K_yearly, dim = 'time')

corrected_covariance_pi = xr.cov(snfr_pi_yearly, corrected_pr_pi, dim = 'time')
corrected_covariance_2K = xr.cov(snfr_2K_yearly, corrected_pr_2K, dim = 'time')
corrected_covariance_4K = xr.cov(snfr_4K_yearly, corrected_pr_4K, dim = 'time')


# variabilities calculated from uncorrected detrended data
variability_prsn_pi_squared = variability_prsn_pi**2
calculated_variability_prsn_pi_squared = variability_snfr_pi**2 * mean_pr_pi**2 + variability_pr_pi**2 * mean_snfr_pi**2 +  variability_snfr_pi**2 * variability_pr_pi**2

# variabilities calculated from corrected detrended data
corrected_variability_prsn_pi_squared = corrected_variability_prsn_pi**2
corrected_calculated_variability_prsn_pi_squared = variability_snfr_pi**2 * corrected_mean_pr_pi**2 + corrected_variability_pr_pi**2 * mean_snfr_pi**2 +  variability_snfr_pi**2 * corrected_variability_pr_pi**2

# without the covariance term
other_calculated = variability_snfr_pi**2 * corrected_mean_pr_pi**2 + corrected_variability_pr_pi**2 * mean_snfr_pi**2

diff_corrected_uncorrected = corrected_variability_prsn_pi_squared - variability_prsn_pi_squared

diff_sig2 = calculated_variability_prsn_pi_squared - variability_prsn_pi_squared
corrected_diff_sig2 = corrected_calculated_variability_prsn_pi_squared - corrected_variability_prsn_pi_squared
meta_diff = corrected_diff_sig2 - diff_sig2
other_diff = other_calculated - corrected_variability_prsn_pi_squared

corrected_cov_diff = corrected_sigma2_pi_cov - corrected_variability_prsn_pi_squared
cov_diff = sigma2_pi_cov - variability_prsn_pi_squared

In [ ]:
proj = ccrs.NorthPolarStereo(central_longitude=0)
fig, ax = plt.subplots(1,4, figsize=(18, 8), subplot_kw={"projection": proj})
standard_2d_plot_polar(corrected_variability_prsn_pi**2, r'$\sigma_{prsn}^2$', cbar_scale = 'log', linthresh = 1e-4, min_scale = 0.0001, ax = ax[0], fig = fig)
standard_2d_plot_polar(variability_snfr_pi**2 * corrected_mean_pr_pi**2 , r'$\sigma_{f}^2$pr$^2$', cbar_scale = 'log', linthresh = 1e-3, min_scale = 1e-5, ax = ax[1], fig = fig)
standard_2d_plot_polar(corrected_variability_pr_pi**2 * mean_snfr_pi**2 , r'$\sigma_{pr}^2$f$^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-5, ax = ax[2], fig = fig)
standard_2d_plot_polar(variability_snfr_pi**2 * corrected_variability_pr_pi**2, r'$\sigma_{f}^2$$\sigma_{pr}^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-7, ax = ax[3], fig = fig)
plt.subplots_adjust(hspace=-0.01, wspace=0.3)
plt.show()

proj = ccrs.NorthPolarStereo(central_longitude=0)
fig, ax = plt.subplots(1,4, figsize=(18, 8), subplot_kw={"projection": proj})
standard_2d_plot_polar(corrected_variability_prsn_2K**2, r'$\sigma_{prsn}^2$', cbar_scale = 'log', linthresh = 1e-4, min_scale = 0.0001, ax = ax[0], fig = fig)
standard_2d_plot_polar(variability_snfr_2K**2 * corrected_mean_pr_2K**2 , r'$\sigma_{f}^2$pr$^2$', cbar_scale = 'log', linthresh = 1e-3, min_scale = 1e-5, ax = ax[1], fig = fig)
standard_2d_plot_polar(corrected_variability_pr_2K**2 * mean_snfr_2K**2 , r'$\sigma_{pr}^2$f$^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-5, ax = ax[2], fig = fig)
standard_2d_plot_polar(variability_snfr_2K**2 * corrected_variability_pr_2K**2, r'$\sigma_{f}^2$$\sigma_{pr}^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-7, ax = ax[3], fig = fig)
plt.subplots_adjust(hspace=-0.01, wspace=0.3)
plt.show()

proj = ccrs.NorthPolarStereo(central_longitude=0)
fig, ax = plt.subplots(1,4, figsize=(18, 8), subplot_kw={"projection": proj})
standard_2d_plot_polar(corrected_variability_prsn_4K**2, r'$\sigma_{prsn}^2$', cbar_scale = 'log', linthresh = 1e-4, min_scale = 0.0001, ax = ax[0], fig = fig)
standard_2d_plot_polar(variability_snfr_4K**2 * corrected_mean_pr_4K**2 , r'$\sigma_{f}^2$pr$^2$', cbar_scale = 'log', linthresh = 1e-3, min_scale = 1e-5, ax = ax[1], fig = fig)
standard_2d_plot_polar(corrected_variability_pr_4K**2 * mean_snfr_4K**2 , r'$\sigma_{pr}^2$f$^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-5, ax = ax[2], fig = fig)
standard_2d_plot_polar(variability_snfr_4K**2 * corrected_variability_pr_4K**2, r'$\sigma_{f}^2$$\sigma_{pr}^2$', cbar_scale = 'log', linthresh = 1e-7, min_scale = 1e-7, ax = ax[3], fig = fig)
plt.subplots_adjust(hspace=-0.01, wspace=0.3)
plt.show()

### Some functions for adjusting and testing dictionaries

In [ ]:
def find_min(dictionary):
    for scenario in dictionary.keys():
        print(float(dictionary[scenario]['prsn'].min()))

find_min(EC_earth_detrended_yearly_data)
find_min(EC_earth_monthly_data)
find_min(UKESM_detrended_yearly_data)
find_min(UKESM_monthly_data)
find_min(CNRM_detrended_yearly_data)
find_min(CNRM_monthly_data)

In [ ]:
def new_pr_and_prsn(pr, prsn, snfr):
    pr_new = pr.where(pr >= 0, 0)
    prsn_new = prsn.where(prsn >= 0, 0)
    
    difference = pr_new-prsn_new
    prsn_overshot = difference.where(difference < 0, 0)
    
    prsn_new = prsn_new + prsn_overshot # make it so snowfall cannot be larger than precipitation
    return pr_new, prsn_new, snfr

def new_pr_and_prsn2(pr, prsn, snfr):
    pr_new = pr.where(pr >= 0, 0)
    prsn_new = prsn.where(prsn >= 0, 0)
    
    difference = pr_new-prsn_new
    prsn_new = prsn_new.where(difference >= 0)
    pr_new = pr_new.where(difference >= 0)
    snfr_new = pr_new.where(difference >= 0)
    
    return pr_new, prsn_new, snfr_new

def create_dictionary_corrected_data(data_dictionary, correction):
    dictionary = {
        'pi': {'tas': data_dictionary['pi']['tas'],
               'prsn': correction(data_dictionary['pi']['pr'], data_dictionary['pi']['prsn'], data_dictionary['pi']['snfr'])[1],
               'pr': correction(data_dictionary['pi']['pr'], data_dictionary['pi']['prsn'], data_dictionary['pi']['snfr'])[0],
               'snfr': correction(data_dictionary['pi']['pr'], data_dictionary['pi']['prsn'], data_dictionary['pi']['snfr'])[2]},
        
        '2K': {'tas': data_dictionary['2K']['tas'],
               'prsn': correction(data_dictionary['2K']['pr'], data_dictionary['2K']['prsn'], data_dictionary['2K']['snfr'])[1],
               'pr': correction(data_dictionary['2K']['pr'], data_dictionary['2K']['prsn'], data_dictionary['2K']['snfr'])[0],
               'snfr': correction(data_dictionary['2K']['pr'], data_dictionary['2K']['prsn'], data_dictionary['2K']['snfr'])[2]},
            
        '4K': {'tas': data_dictionary['4K']['tas'],
               'prsn': correction(data_dictionary['4K']['pr'], data_dictionary['4K']['prsn'], data_dictionary['4K']['snfr'])[1],
               'pr': correction(data_dictionary['4K']['pr'], data_dictionary['4K']['prsn'], data_dictionary['4K']['snfr'])[0],
               'snfr': correction(data_dictionary['4K']['pr'], data_dictionary['4K']['prsn'], data_dictionary['4K']['snfr'])[2]}}
    return dictionary

yearly_data = create_dictionary_data(12, 'non')

corrected_detrended_yearly_data = create_dictionary_corrected_data(detrended_yearly_data, new_pr_and_prsn)
corrected_yearly_data = create_dictionary_corrected_data(yearly_data, new_pr_and_prsn)

corrected_detrended_yearly_data2 = create_dictionary_corrected_data(detrended_yearly_data, new_pr_and_prsn2)
corrected_yearly_data2 = create_dictionary_corrected_data(yearly_data, new_pr_and_prsn2)

In [ ]:

def yearly_mean(data_array, snfr_threshold_snow = 0, snfr_threshold_pr = 0):

    if isinstance(data_array, (xr.DataArray, xr.Dataset)):      
           
        t0 = data_array.time.values[0]
        
        if isinstance(t0, cftime.Datetime360Day):
            yearly_array = data_array.resample(time="YS").mean(dim='time')
            print('360 calender, normal mean taken')
            return yearly_array
            
        else: 
            start_year = int(data_array.time.min().dt.year)
            end_year = int(data_array.time.max().dt.year)
            years = np.linspace(start_year, end_year, end_year - start_year + 1)
            times = []
            means = []
            for year in years:
                year_data = data_array.sel(time=data_array.time.dt.year == year)
            
                if isinstance(t0, np.datetime64):
                    times.append(year_data.time.mean().values)
                elif isinstance(t0, cftime.datetime):
                    times.append(year_data.time.mean().item())
                else:
                    print(f"Unknown time type: {type(t0)}")
                
                 # --- Compute days-per-month weights ---
                days_in_month = year_data.time.dt.days_in_month
                weights = days_in_month / days_in_month.sum()
                # --- Compute weighted time mean ---
                timmean = year_data.weighted(weights).mean('time')#.to_dataset(name=variable)
                means.append(timmean)
            result = xr.concat(means, dim='time')
            result = result.assign_coords(time=("time", times))
            # print(type(result.time.values[0]))
            return result

    elif isinstance(data_array, dict):

        if 'snfr'in data_array.keys():
            new_dict = {}

            for k, v in data_array.items():
                if k == 'snfr':
                    continue
                    
                new_dict[k] = yearly_mean(v, snfr_threshold_snow, snfr_threshold_pr)

            new_dict['snfr'] = snow_fraction(new_dict['prsn'], new_dict['pr'], snfr_threshold_snow, snfr_threshold_pr)

            return new_dict

        else:
            return {k: yearly_mean(v, snfr_threshold_snow, snfr_threshold_pr) for k, v in data_array.items()}

    # --- lists (your variability output) ---
    elif isinstance(data_array, list):
        return [yearly_mean(v, snfr_threshold_snow, snfr_threshold_pr) for v in data_array]

    else:
        print('Data is not a dictionary, list, xarray or xr dataset. Nothing was changed.')
        return data_array